In [1]:
!pip install -q -U bitsandbytes>=0.43.1 transformers>=4.41.0 accelerate peft open_clip_torch torchvision wandb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuda-python 12.9.4 requires cuda-bindings~=12.9.4, but you have cuda-bindings 13.2.0 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
libcuvs-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
libraft-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, b

In [2]:
!git clone https://github.com/Turab-Zaidi/OmniMed.git
%cd /kaggle/working/OmniMed




Cloning into 'OmniMed'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 114 (delta 73), reused 81 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 15.77 KiB | 2.25 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/kaggle/working/OmniMed


In [3]:
import sys
import torch
from unittest.mock import MagicMock
import os

mock_triton = MagicMock()
sys.modules["triton"] = mock_triton
sys.modules["triton.ops"] = mock_triton
sys.modules["triton.ops.matmul_perf_model"] = mock_triton

os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:/usr/local/nvidia/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ['BNB_CUDA_VERSION'] = '121'

try:
    import bitsandbytes as bnb
    print(f" Bitsandbytes {bnb.__version__} loaded successfully!")
    print(f" CUDA Available: {torch.cuda.is_available()}")
except Exception as e:
    print(f"❌ Failed to load bitsandbytes: {e}")

 Bitsandbytes 0.49.2 loaded successfully!
 CUDA Available: True


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

In [5]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: turab-z567 (turab-z567-jamia-millia-islamia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:


%env HF_TOKEN={HF_TOKEN}
%env HF_HOME=/kaggle/temp/huggingface_cache

import os
os.makedirs('/kaggle/temp/huggingface_cache', exist_ok=True)



%cd /kaggle/working/OmniMed

!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && \
 export PYTHONPATH=$PYTHONPATH:/kaggle/working/OmniMed && \
 accelerate launch \
    --multi_gpu \
    --num_processes 2 \
    --num_machines 1 \
    --mixed_precision fp16 \
    src/trainer.py

env: HF_TOKEN=hf_***REDACTED***
env: HF_HOME=/kaggle/temp/huggingface_cache
/kaggle/working/OmniMed
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
config.json: 100%|█████████████████████████████| 855/855 [00:00<00:00, 2.92MB/s]
tokenizer_config.json: 55.4kB [00:00, 91.4MB/s]
tokenizer.json: 9.09MB [00:00, 24.0MB/s]
open_clip_config.json: 100%|███████████████████| 707/707 [00:00<00:00, 2.50MB/s]
open_clip_pytorch_model.bin: 100%|████████████| 784M/784M [00:03<00:00, 205MB/s]
config.json: 100%|█████████████████████████████| 385/385 [00:00<00:00, 1.86MB/s]
This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=

This can be used 

In [7]:
from huggingface_hub import HfApi
api = HfApi()

try:
    api.model_info("meta-llama/Llama-3.1-8B-Instruct")
    print("✅ Success: Your token has access to this model!")
except Exception as e:
    print(f"❌ Error: Your token does NOT have access. Check permissions: {e}")

✅ Success: Your token has access to this model!
